# Data Pipeline

In this lesson, we will learn how to load a dataset and prepare it for training using PyTorch's `DataLoader`.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Loading the Dataset

We will use the `cosmopedia-v2` dataset in streaming mode to handle large amounts of data efficiently.

In [ ]:
dataset = load_dataset(
    "HuggingFaceTB/smollm-corpus",
    "cosmopedia-v2",
    split="train",
    streaming=True
)

# View a sample
sample = next(iter(dataset))
print(sample["text"][:200])

## Tokenizing and Batching

We need to tokenize the text and group it into batches.

In [ ]:
def collate_fn(batch):
    texts = [item["text"] for item in batch]
    encodings = tokenizer(
        texts, 
        truncation=True, 
        padding=True, 
        max_length=512, 
        return_tensors="pt"
    )
    return encodings

dataloader = DataLoader(dataset, batch_size=4, collate_fn=collate_fn)

# Fetch a batch
batch = next(iter(dataloader))
print(f"Input IDs shape: {batch['input_ids'].shape}")